In [11]:
# =========================================================================
# CÉLULA 1: CARREGAMENTO DE DADOS (AJUSTADO PARA A SUA PASTA)
# =========================================================================
import pandas as pd
import os

print("--- FASE 1: CARREGAMENTO ---")

# Como você está em C:\Users\User e a pasta Data está lá também:
BASE_PATH = "Data" 

# Construindo os caminhos
FILE_PATH_1 = os.path.join(BASE_PATH, "Base Artigo 1.csv")
FILE_PATH_2 = os.path.join(BASE_PATH, "Base Artigo 2.csv")
FILE_PATH_3 = os.path.join(BASE_PATH, "Base Artigo 3.csv")

# Carregar os três arquivos CSV
try:
    df1 = pd.read_csv(FILE_PATH_1, sep=',')
    df2 = pd.read_csv(FILE_PATH_2, sep=',')
    df3 = pd.read_csv(FILE_PATH_3, sep=',')
    print("Arquivos carregados com sucesso!")
    load_successful = True
except FileNotFoundError as e:
    print(f"ERRO: Não encontrei o ficheiro. Verifique se ele está dentro da pasta Data.")
    print(f"Detalhe do erro: {e}")

--- FASE 1: CARREGAMENTO ---
Arquivos carregados com sucesso!


In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

print("--- FASE 2: PRÉ-PROCESSAMENTO (CORREÇÃO BINÁRIA) ---")

# 1. Padronizar e Fundir
df1_prep = df1.rename(columns={'Numero': 'Atleta_ID'}).set_index('Atleta_ID')
df2_prep = df2.rename(columns={'Número': 'Atleta_ID'}).set_index('Atleta_ID')
df3_prep = df3.rename(columns={'Número': 'Atleta_ID'}).set_index('Atleta_ID')

df_merge = df1_prep.join(df2_prep, how='outer', lsuffix='_art1', rsuffix='_art2')
df_final = df_merge.join(df3_prep, how='outer', rsuffix='_art3')

# 2. CONVERSÃO BINÁRIA CRÍTICA
# Os teus dados têm 1 e 2. Vamos transformar 1 em 0 (Saudável) e 2 em 1 (Entorse)
y_raw = df_final['Entorse_art1'].fillna(df_final['Entorse']).fillna(df_final['Entorse_art2'])
y = y_raw.map({1: 0, 2: 1}).fillna(0).astype(int)

# 3. Limpeza de Features
X = df_final.drop(columns=[col for col in df_final.columns if 'Entorse' in col or 'Lado' in col])
X = X.select_dtypes(include=[np.number])
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna(axis=1, how='all')

# 4. Imputação e Divisão
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.75, random_state=42, stratify=y)

# 5. Normalização
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset Binário pronto! Classes detetadas: {np.unique(y)}")
print(f"Total de amostras: {len(y)} | Variáveis: {X_train_scaled.shape[1]}")

--- FASE 2: PRÉ-PROCESSAMENTO (CORREÇÃO BINÁRIA) ---
Dataset Binário pronto! Classes detetadas: [0 1]
Total de amostras: 61 | Variáveis: 244


In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

print("--- FASE 3: CONSTRUÇÃO DA REDE NEURONAL ---")

# 1. Definir a arquitetura
model = Sequential([
    # Camada de Entrada: ajusta-se automaticamente ao número de variáveis resultantes do merge
    Input(shape=(X_train_scaled.shape[1],)), 
    
    # Camada 1: 64 neurónios para captar interações complexas
    Dense(64, activation='relu'),
    Dropout(0.3), # Previne overfitting (importante para o seu artigo)
    
    # Camada 2: 32 neurónios
    Dense(32, activation='relu'),
    Dropout(0.2),
    
    # Camada 3: 16 neurónios para refinar a decisão
    Dense(16, activation='relu'),
    
    # Camada de Saída: Sigmoid para dar a probabilidade de entorse (0 a 1)
    Dense(1, activation='sigmoid')
])

# 2. Compilação
# Usamos 'Recall' como métrica principal porque, em saúde, falhar uma lesão é o pior erro.
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
)

model.summary()

--- FASE 3: CONSTRUÇÃO DA REDE NEURONAL ---


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                     │ (None, 64)                  │          15,680 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_13 (Dense)                     │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_14 (Dense)                     │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_15 (Dense)                     │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 18,305 (71.50 KB)

 Trainable params: 18,305 (71.50 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
from tensorflow.keras.callbacks import EarlyStopping
import time

print("--- FASE 4: TREINO DA REDE NEURONAL ---")
print("A treinar o modelo... Por favor, aguarde.")

# Configurar paragem antecipada para evitar overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    mode='min',
    patience=15,
    restore_best_weights=True
)

# ⏱️ INÍCIO DA TEMPORIZAÇÃO
start_time = time.time()

history = model.fit(
    X_train_scaled, y_train,
    epochs=150,
    batch_size=16,
    validation_data=(X_test_scaled, y_test),
    callbacks=[early_stop],
    verbose=0
)

# ⏱️ FIM DA TEMPORIZAÇÃO
end_time = time.time()

execution_time_ms = (end_time - start_time) * 1000

print("Treino concluído com sucesso!")
print(f"Tempo de treino da Rede Neuronal: {execution_time_ms:.2f} ms")


--- FASE 4: TREINO DA REDE NEURONAL ---
A treinar o modelo... Por favor, aguarde.
Treino concluído com sucesso!
Tempo de treino da Rede Neuronal: 4885.78 ms


In [15]:
from sklearn.metrics import recall_score, f1_score, accuracy_score, confusion_matrix
import time

print("--- FASE 5: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (REDE NEURONAL) ---")

# ⏱️ Início da medição do tempo
start_time = time.perf_counter()

# 1. Obter previsões
y_pred_probs = model.predict(X_test_scaled, verbose=0)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# 2. Calcular Métricas
acc = accuracy_score(y_test, y_pred)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)
fn = cm[1, 0] if cm.shape == (2, 2) else 0  # Segurança

# ⏱️ Fim da medição do tempo
end_time = time.perf_counter()
execution_time_ms = (end_time - start_time) * 1000

# 3. Output Formatado (estilo artigo)
print(f"Precisão (Accuracy): {acc:.4f}")
print(f"Recall (Sensibilidade): {rec:.4f}")
print(f"F-Score: {f1:.4f}")
print(f"Tempo de Execução: {execution_time_ms:.2f} ms")

print("\nMatriz de Confusão:")
print(cm)

print(f"\nFalsos Negativos (FN - Entorse real não prevista): {fn}")

# 4. Conclusão do Modelo
if rec > 0.90:
    print("\n--- ✅ OTIMIZAÇÃO CONCLUÍDA: RECALL ELEVADO ---")
else:
    print("\n--- ⚠️ RECALL BAIXO: É NECESSÁRIO AJUSTAR A REDE NEURONAL ---")
from sklearn.metrics import recall_score, f1_score, accuracy_score, confusion_matrix
import time

print("--- FASE 5: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (REDE NEURONAL) ---")

# ⏱️ Início da medição do tempo
start_time = time.perf_counter()

# 1. Obter previsões
y_pred_probs = model.predict(X_test_scaled, verbose=0)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# 2. Calcular Métricas
acc = accuracy_score(y_test, y_pred)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)
fn = cm[1, 0] if cm.shape == (2, 2) else 0  # Segurança

# ⏱️ Fim da medição do tempo
end_time = time.perf_counter()
execution_time_ms = (end_time - start_time) * 1000

# 3. Output Formatado (estilo artigo)
print(f"Precisão (Accuracy): {acc:.4f}")
print(f"Recall (Sensibilidade): {rec:.4f}")
print(f"F-Score: {f1:.4f}")
print(f"Tempo de Execução: {execution_time_ms:.2f} ms")

print("\nMatriz de Confusão:")
print(cm)

print(f"\nFalsos Negativos (FN - Entorse real não prevista): {fn}")

# 4. Conclusão do Modelo
if rec > 0.90:
    print("\n--- ✅ OTIMIZAÇÃO CONCLUÍDA: RECALL ELEVADO ---")
else:
    print("\n--- ⚠️ RECALL BAIXO: É NECESSÁRIO AJUSTAR A REDE NEURONAL ---")


--- FASE 5: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (REDE NEURONAL) ---
Precisão (Accuracy): 0.5000
Recall (Sensibilidade): 0.6923
F-Score: 0.6102
Tempo de Execução: 246.74 ms

Matriz de Confusão:
[[ 5 15]
 [ 8 18]]

Falsos Negativos (FN - Entorse real não prevista): 8

--- ⚠️ RECALL BAIXO: É NECESSÁRIO AJUSTAR A REDE NEURONAL ---
--- FASE 5: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (REDE NEURONAL) ---
Precisão (Accuracy): 0.5000
Recall (Sensibilidade): 0.6923
F-Score: 0.6102
Tempo de Execução: 104.09 ms

Matriz de Confusão:
[[ 5 15]
 [ 8 18]]

Falsos Negativos (FN - Entorse real não prevista): 8

--- ⚠️ RECALL BAIXO: É NECESSÁRIO AJUSTAR A REDE NEURONAL ---


In [ ]:
%pip install tensorflow